# Hotel Booking Analysis

**Author:** Hotel Booking Analysis Project  
**Dataset:** `hotel_bookings.csv`  
**Cleaned Dataset:** `hotel_bookings_cleaned.csv`  
**Python Version:** 3.13  
**Period Covered:** 2015 – 2017

---

## 1. Problem Statement

The hotel industry faces significant revenue losses from booking cancellations, seasonal demand fluctuations, and over-reliance on third-party booking channels. This project analyses a real-world hotel booking dataset to:

- Understand booking patterns across two hotel types (City Hotel and Resort Hotel)
- Identify the scale and drivers of booking cancellations
- Examine pricing (ADR), stay duration, and guest loyalty trends
- Derive actionable business insights to improve revenue management

**Business Questions Answered:**
1. What is the total number of bookings?
2. How many bookings are for City Hotel vs Resort Hotel?
3. What percentage of bookings were cancelled?
4. Which hotel type has the higher cancellation rate?
5. Which months have the highest and lowest number of bookings?
6. What is the average length of stay?
7. What is the average daily rate (ADR)?
8. Which market segment has the most bookings?
9. What percentage of bookings are from repeated guests?
10. Is there a relationship between lead time and cancellation?

## 2. Dataset Description

| Attribute | Value |
|---|---|
| Source file | `hotel_bookings.csv` |
| Raw rows | 119,390 |
| Columns | 33 |
| Hotel types | Resort Hotel, City Hotel |
| Years | 2015, 2016, 2017 |

**Key columns:**

| Column | Description |
|---|---|
| `hotel` | Hotel type: Resort Hotel or City Hotel |
| `is_canceled` | 1 = cancelled, 0 = not cancelled |
| `lead_time` | Days between booking and arrival |
| `arrival_date_year/month` | Arrival year and month |
| `stays_in_weekend_nights` | Weekend nights booked |
| `stays_in_week_nights` | Weekday nights booked |
| `adults / children / babies` | Guest counts |
| `meal` | Meal plan (BB, HB, FB, SC) |
| `country` | Guest country of origin |
| `market_segment` | Booking channel (Online TA, Direct, etc.) |
| `adr` | Average Daily Rate (EUR) |
| `reservation_status` | Check-Out, Canceled, No-Show |
| `agent` | Travel agent ID (blank = no agent) |
| `company` | Company ID (blank = no company) |

## 3. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# Shared style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
    'axes.labelsize':   11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       120,
})

# Colour palette
C_BLUE, C_PURPLE, C_RED, C_GREEN = '#3B82D4', '#7C5CD8', '#E05252', '#3BA876'
PALETTE_2 = [C_BLUE, C_PURPLE]

print('pandas  :', pd.__version__)
print('numpy   :', np.__version__)
print('matplotlib:', matplotlib.__version__)
print('seaborn :', sns.__version__)

## 4. Load Raw Dataset

In [ ]:
df_raw = pd.read_csv('hotel_bookings.csv', low_memory=False)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 5. Data Inspection

In [ ]:
print('=== Shape ===')
print(f'Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}')
print()
print('=== Data Types ===')
print(df_raw.dtypes)

In [ ]:
print('=== Missing Values ===')
missing = df_raw.isnull().sum()
missing = missing[missing > 0]
print(missing)
print()
print('=== Missing % ===')
print((missing / len(df_raw) * 100).round(2))

In [ ]:
print('=== Duplicate Rows (excluding index column) ===')
cols = [c for c in df_raw.columns if c != 'index']
dupes = df_raw.duplicated(subset=cols).sum()
print(f'Duplicate rows: {dupes:,}')

In [ ]:
print('=== Descriptive Statistics (numeric) ===')
df_raw.describe().round(2)

In [ ]:
print('=== Unique values in key categorical columns ===')
for col in ['hotel', 'arrival_date_year', 'meal', 'market_segment',
            'distribution_channel', 'reservation_status', 'customer_type', 'deposit_type']:
    print(f'{col:35s}: {df_raw[col].unique().tolist()}')

In [ ]:
# Show mixed date formats in reservation_status_date
print('=== Date format samples ===')
print(df_raw['reservation_status_date'].head(15).tolist())

## 6. Data Cleaning

The following issues were identified and corrected. **The original `hotel_bookings.csv` is never modified.**

In [ ]:
df = df_raw.copy()

# ── 6.1 Fill missing values ───────────────────────────────────────────────
# agent / company: blank = no agent / no company, fill with 0
agent_missing   = df['agent'].isna().sum()
company_missing = df['company'].isna().sum()
country_missing = df['country'].isna().sum()
children_missing= df['children'].isna().sum()

df['agent']    = df['agent'].fillna(0).astype(int)
df['company']  = df['company'].fillna(0).astype(int)
df['country']  = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0).astype(int)

print(f'agent    : {agent_missing:,} blanks filled with 0')
print(f'company  : {company_missing:,} blanks filled with 0')
print(f'country  : {country_missing} blanks filled with "Unknown"')
print(f'children : {children_missing} blanks filled with 0')

In [ ]:
# ── 6.2 Remove duplicate rows ────────────────────────────────────────────
cols_for_dedup = [c for c in df.columns if c != 'index']
before = len(df)
df = df.drop_duplicates(subset=cols_for_dedup, keep='first')
print(f'Duplicates removed: {before - len(df):,}  ({before:,} -> {len(df):,} rows)')

In [ ]:
# ── 6.3 Standardise reservation_status_date ──────────────────────────────
# Two formats: D/M/YYYY and DD-MM-YY (European day-first in both cases)
def parse_mixed_dates(series):
    def _parse_one(val):
        if pd.isna(val) or str(val) == 'nan':
            return pd.NaT
        val = str(val).strip()
        if '-' in val:
            parts = val.split('-')
            if len(parts) == 3 and len(parts[2]) == 2:
                parts[2] = '20' + parts[2]
            val = '/'.join(parts)
        try:
            return pd.to_datetime(val, dayfirst=True)
        except Exception:
            return pd.NaT
    return series.map(_parse_one)

df['reservation_status_date'] = parse_mixed_dates(df['reservation_status_date'])
nat_count = df['reservation_status_date'].isna().sum()
print(f'reservation_status_date parsed to datetime  (NaT remaining: {nat_count})')

In [ ]:
# ── 6.4 Fix negative ADR ─────────────────────────────────────────────────
neg_adr = (df['adr'] < 0).sum()
df.loc[df['adr'] < 0, 'adr'] = 0
print(f'Negative ADR rows fixed: {neg_adr}')

# ── 6.5 Flag zero-ADR non-complementary Check-Outs ───────────────────────
zero_adr_mask = (
    (df['adr'] == 0) &
    (df['reservation_status'] == 'Check-Out') &
    (df['market_segment'] != 'Complementary')
)
df['is_zero_adr'] = zero_adr_mask.astype(int)
print(f'is_zero_adr flag set for {zero_adr_mask.sum():,} rows')

In [ ]:
# ── 6.6 Replace meal == "Undefined" with "SC" ─────────────────────────────
undef_meal = (df['meal'] == 'Undefined').sum()
df['meal'] = df['meal'].replace('Undefined', 'SC')
print(f'meal "Undefined" replaced with "SC": {undef_meal:,} rows')

In [ ]:
# ── 6.7 Drop zero-guest Check-Out rows ───────────────────────────────────
df['_total_guests'] = df['adults'] + df['children'] + df['babies']
zero_guest = ((df['_total_guests'] == 0) & (df['reservation_status'] == 'Check-Out')).sum()
df = df[~((df['_total_guests'] == 0) & (df['reservation_status'] == 'Check-Out'))]
print(f'Zero-guest Check-Out rows dropped: {zero_guest}')

# ── 6.8 Drop zero-night Check-Out rows ───────────────────────────────────
df['_total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
zero_nights = ((df['_total_nights'] == 0) & (df['reservation_status'] == 'Check-Out')).sum()
df = df[~((df['_total_nights'] == 0) & (df['reservation_status'] == 'Check-Out'))]
print(f'Zero-night Check-Out rows dropped: {zero_nights}')

df = df.drop(columns=['_total_guests', '_total_nights'])

In [ ]:
# ── 6.9 Fix is_canceled vs reservation_status mismatch ───────────────────
status_map = {'Check-Out': 0, 'Canceled': 1, 'No-Show': 1}
expected = df['reservation_status'].map(status_map)
mismatch = (expected.notna() & (df['is_canceled'] != expected)).sum()
df.loc[expected.notna() & (df['is_canceled'] != expected), 'is_canceled'] = \
    expected[expected.notna() & (df['is_canceled'] != expected)]
print(f'is_canceled mismatches corrected: {mismatch}')

# ── 6.10 Flag group-block rows (adults > 10) ─────────────────────────────
df['is_group_block'] = (df['adults'] > 10).astype(int)
print(f'is_group_block flagged: {df["is_group_block"].sum()} rows')

# ── Summary ───────────────────────────────────────────────────────────────
print(f'\nFinal cleaned shape: {df.shape}')
print(f'Remaining nulls    : {df.isnull().sum().sum()}')

In [ ]:
# Load the pre-saved cleaned dataset for the rest of the analysis
# (identical to the df we just built above)
df = pd.read_csv('hotel_bookings_cleaned.csv', low_memory=False)
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
print(f'Cleaned dataset loaded: {df.shape}')
df.head()

## 7. Exploratory Data Analysis

### Q1 — What is the total number of bookings?

In [ ]:
total = len(df)
print(f'Total bookings (cleaned dataset): {total:,}')
# Result: 86,678

### Q2 — How many bookings are for City Hotel and Resort Hotel?

In [ ]:
hotel_counts = df['hotel'].value_counts()
hotel_pct    = (hotel_counts / len(df) * 100).round(2)
print(pd.DataFrame({'Count': hotel_counts, 'Pct (%)': hotel_pct}))
# City Hotel: 53,070 (61.23%)  |  Resort Hotel: 33,608 (38.77%)

### Q3 — What percentage of bookings were cancelled?

In [ ]:
canceled_count = df['is_canceled'].sum()
cancel_rate    = df['is_canceled'].mean() * 100
print(f'Cancelled bookings : {canceled_count:,}')
print(f'Cancellation rate  : {cancel_rate:.2f}%')
# 24,025 bookings cancelled  |  27.72%

### Q4 — Which hotel type has the higher cancellation rate?

In [ ]:
cancel_by_hotel = (
    df.groupby('hotel')['is_canceled']
      .mean().mul(100).round(2)
)
print(cancel_by_hotel)
# City Hotel: 30.24%  |  Resort Hotel: 23.73%

### Q5 — Which months have the highest and lowest number of bookings?

In [ ]:
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
monthly = df['arrival_date_month'].value_counts().reindex(MONTH_ORDER)
print('Bookings per month:')
print(monthly.to_string())
print(f'\nPeak  : {monthly.idxmax()} ({monthly.max():,})')
print(f'Lowest: {monthly.idxmin()} ({monthly.min():,})')
# Peak: August (11,195)  |  Lowest: January (4,642)

### Q6 — What is the average length of stay?

In [ ]:
avg_overall = df['total_nights'].mean().round(2)
avg_by_hotel = df.groupby('hotel')['total_nights'].mean().round(2)
print(f'Overall avg stay : {avg_overall} nights')
print(avg_by_hotel)
# Overall: 3.65  |  City: 3.15  |  Resort: 4.44

### Q7 — What is the average daily rate (ADR)?

In [ ]:
EUR_TO_INR = 90
df_adr = df[df['is_zero_adr'] == 0]
print(f'Mean ADR   : €{df_adr["adr"].mean():.2f}  (₹{df_adr["adr"].mean()*EUR_TO_INR:,.0f})')
print(f'Median ADR : €{df_adr["adr"].median():.2f}  (₹{df_adr["adr"].median()*EUR_TO_INR:,.0f})')
print(f'Max ADR    : €{df_adr["adr"].max():.2f}  (₹{df_adr["adr"].max()*EUR_TO_INR:,.0f})')
print()
print('Mean ADR by hotel type:')
adr_hotel = df_adr.groupby('hotel')['adr'].mean().round(2)
for h, v in adr_hotel.items():
    print(f'  {h}: €{v:.2f}  (₹{v*EUR_TO_INR:,.0f})')
# Overall mean: €107.65 (₹9,689)  |  City: €112.18 (₹10,096)  |  Resort: €100.49 (₹9,044)

### Q8 — Which market segment has the most bookings?

In [ ]:
seg = df['market_segment'].value_counts()
seg_pct = (seg / len(df) * 100).round(2)
print(pd.DataFrame({'Count': seg, 'Pct (%)': seg_pct}))
# Online TA: 51,300 (59.18%)  — dominates by a wide margin

### Q9 — What percentage of bookings are from repeated guests?

In [ ]:
repeat_count = df['is_repeated_guest'].sum()
repeat_pct   = df['is_repeated_guest'].mean() * 100
print(f'Repeated guests : {repeat_count:,}')
print(f'Repeat rate     : {repeat_pct:.2f}%')
# 3,147 bookings  |  3.63%

### Q10 — Is there a relationship between lead time and cancellation?

In [ ]:
# Pearson correlation
corr = df['lead_time'].corr(df['is_canceled']).round(4)
print(f'Pearson r (lead_time, is_canceled) = {corr}')

# Mean lead time by outcome
mean_lead = df.groupby('is_canceled')['lead_time'].mean().round(2)
print(f'Mean lead time — Not cancelled : {mean_lead[0]} days')
print(f'Mean lead time — Cancelled     : {mean_lead[1]} days')

# Binned cancellation rates
bins   = [0, 7, 30, 90, 180, 365, df['lead_time'].max()+1]
labels = ['0-7d', '8-30d', '31-90d', '91-180d', '181-365d', '365d+']
df['lead_bin'] = pd.cut(df['lead_time'], bins=bins, labels=labels, right=True)
cancel_by_bin  = (
    df.groupby('lead_bin', observed=True)['is_canceled']
      .agg(['mean','count'])
      .assign(cancel_rate=lambda x: (x['mean']*100).round(2))
)
print('\nCancellation rate by lead-time bucket:')
print(cancel_by_bin[['count','cancel_rate']].to_string())
# r = 0.183  |  cancelled avg 105.72 days vs 70.52 for completed
# Rate rises from 9.77% (0-7 days) to 41.10% (365+ days)

## 8. Visualizations

### 8.1 Bookings by Hotel Type

In [ ]:
hc  = df['hotel'].value_counts()
pct = (hc / hc.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Bookings by Hotel Type', fontsize=14, fontweight='bold')

# Donut
axes[0].pie(hc, labels=hc.index, colors=PALETTE_2, autopct='%1.1f%%',
            startangle=90, wedgeprops=dict(width=0.45, edgecolor='white'))
axes[0].set_title('Share')

# Bar
bars = axes[1].bar(hc.index, hc.values, color=PALETTE_2, width=0.45, edgecolor='white')
for bar, val in zip(bars, hc.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                 f'{val:,}', ha='center', fontweight='bold')
axes[1].set_ylabel('Number of Bookings')
axes[1].set_title('Absolute Count')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
axes[1].set_ylim(0, hc.max()*1.15)

plt.tight_layout()
plt.show()

### 8.2 Monthly Booking Trend

In [ ]:
monthly = df['arrival_date_month'].value_counts().reindex(MONTH_ORDER).fillna(0)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(range(12), monthly.values, color=C_BLUE, alpha=0.8, edgecolor='white')
ax.plot(range(12), monthly.values, color=C_BLUE, linewidth=2.5, marker='o', markersize=5)

peak_i = monthly.values.argmax()
low_i  = monthly.values.argmin()
ax.annotate(f'Peak\n{int(monthly.values[peak_i]):,}',
            xy=(peak_i, monthly.values[peak_i]), xytext=(0,14),
            textcoords='offset points', ha='center', fontsize=9,
            fontweight='bold', color=C_BLUE)
ax.annotate(f'Low\n{int(monthly.values[low_i]):,}',
            xy=(low_i, monthly.values[low_i]), xytext=(0,-28),
            textcoords='offset points', ha='center', fontsize=9,
            fontweight='bold', color=C_RED)

ax.set_xticks(range(12))
ax.set_xticklabels([m[:3] for m in MONTH_ORDER], rotation=0)
ax.set_title('Monthly Booking Trend (All Years Combined)')
ax.set_ylabel('Number of Bookings')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 8.3 Cancellation Status

In [ ]:
cancel_vals = df['is_canceled'].map({0:'Not Cancelled', 1:'Cancelled'}).value_counts()
cancel_pct  = (cancel_vals / cancel_vals.sum() * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Cancellation Status', fontsize=14, fontweight='bold')

axes[0].pie(cancel_vals, labels=cancel_vals.index, colors=[C_GREEN, C_RED],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(width=0.45, edgecolor='white'))
axes[0].set_title('Share')

bars = axes[1].barh(cancel_vals.index, cancel_vals.values,
                    color=[C_GREEN, C_RED], edgecolor='white', height=0.45)
for bar, val, pct in zip(bars, cancel_vals.values, cancel_pct.values):
    axes[1].text(bar.get_width()+300, bar.get_y()+bar.get_height()/2,
                 f'{val:,}  ({pct}%)', va='center', fontweight='bold')
axes[1].set_xlabel('Number of Bookings')
axes[1].set_xlim(0, cancel_vals.max()*1.22)
axes[1].set_title('Absolute Count')
plt.tight_layout()
plt.show()

### 8.4 Cancellation Rate by Hotel Type

In [ ]:
cr       = df.groupby('hotel')['is_canceled'].mean().mul(100).round(2)
overall  = df['is_canceled'].mean() * 100

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(cr.index, cr.values, color=PALETTE_2, width=0.45, edgecolor='white')
for bar, val in zip(bars, cr.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{val}%', ha='center', fontsize=13, fontweight='bold')
ax.axhline(overall, linestyle='--', color='#1f2328', linewidth=1.2,
           label=f'Overall avg: {overall:.1f}%')
ax.legend(frameon=False)
ax.set_title('Cancellation Rate by Hotel Type')
ax.set_ylabel('Cancellation Rate (%)')
ax.set_ylim(0, cr.max() * 1.3)
plt.tight_layout()
plt.show()

### 8.5 Bookings by Market Segment

In [ ]:
seg     = df['market_segment'].value_counts().sort_values(ascending=True)
seg_pct = (seg / len(df) * 100).round(1)
colors  = [C_BLUE if s == 'Online TA' else '#90b4e8' for s in seg.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(seg.index, seg.values, color=colors, edgecolor='white', height=0.6)
for bar, val, pct in zip(bars, seg.values, seg_pct.values):
    ax.text(bar.get_width()+200, bar.get_y()+bar.get_height()/2,
            f'{val:,}  ({pct}%)', va='center', fontsize=9.5)
ax.set_title('Bookings by Market Segment')
ax.set_xlabel('Number of Bookings')
ax.set_xlim(0, seg.max()*1.28)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 8.6 Average Daily Rate (ADR) by Hotel Type

In [ ]:
EUR_TO_INR = 90
df_adr     = df[(df['is_zero_adr'] == 0) & (df['adr'] <= 500)].copy()
df_adr['adr_inr'] = df_adr['adr'] * EUR_TO_INR

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
fig.suptitle('Average Daily Rate (ADR) by Hotel Type', fontsize=14, fontweight='bold')

sns.boxplot(data=df_adr, x='hotel', y='adr_inr', hue='hotel',
            palette=PALETTE_2, legend=False,
            width=0.45,
            flierprops=dict(marker='.', alpha=0.3, markersize=3),
            ax=axes[0])
for i, hotel in enumerate(['City Hotel', 'Resort Hotel']):
    m = df_adr.loc[df_adr['hotel']==hotel, 'adr_inr'].mean()
    axes[0].text(i, m+150, f'Mean\n₹{m:,.0f}', ha='center', fontsize=8.5, color='white',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=PALETTE_2[i], edgecolor='none', alpha=0.9))
axes[0].set_title('Distribution (Boxplot)')
axes[0].set_ylabel('ADR (₹)')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{int(x):,}'))

sns.violinplot(data=df_adr, x='hotel', y='adr_inr', hue='hotel',
               palette=PALETTE_2, legend=False, inner='quartile', cut=0, ax=axes[1])
axes[1].set_title('Density (Violin, ADR ≤ ₹45,000)')
axes[1].set_ylabel('ADR (₹)')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{int(x):,}'))

plt.tight_layout()
plt.show()

### 8.7 Average Length of Stay by Hotel Type

In [ ]:
import numpy as np
hotels         = ['City Hotel', 'Resort Hotel']
weekday_avg    = df.groupby('hotel')['stays_in_week_nights'].mean()
weekend_avg    = df.groupby('hotel')['stays_in_weekend_nights'].mean()
total_avg      = df.groupby('hotel')['total_nights'].mean()

x, w = np.arange(2), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x-w/2, [weekday_avg[h] for h in hotels], width=w,
               label='Weekday Nights', color=C_BLUE, edgecolor='white')
bars2 = ax.bar(x+w/2, [weekend_avg[h] for h in hotels], width=w,
               label='Weekend Nights', color=C_PURPLE, edgecolor='white')

for bar in list(bars1)+list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for i, h in enumerate(hotels):
    ax.text(i, max(weekday_avg[h], weekend_avg[h])+0.2,
            f'Total {total_avg[h]:.2f}n', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(hotels)
ax.set_ylabel('Average Nights')
ax.set_title('Average Length of Stay by Hotel Type')
ax.set_ylim(0, max(weekday_avg.values)*1.45)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### 8.8 Lead Time vs Cancellation Rate

In [ ]:
bins   = [0, 7, 30, 90, 180, 365, df['lead_time'].max()+1]
labels = ['0-7', '8-30', '31-90', '91-180', '181-365', '365+']
df['lead_bin'] = pd.cut(df['lead_time'], bins=bins, labels=labels, right=True)
bs = (df.groupby('lead_bin', observed=True)['is_canceled']
        .agg(['mean','count'])
        .assign(rate=lambda x: (x['mean']*100).round(2)))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(bs.index, bs['count'], color=C_BLUE, alpha=0.65, edgecolor='white',
        label='Number of Bookings')
ax1.set_ylabel('Number of Bookings', color=C_BLUE)
ax1.tick_params(axis='y', labelcolor=C_BLUE)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

ax2 = ax1.twinx()
ax2.plot(bs.index, bs['rate'], color=C_RED, linewidth=2.5,
         marker='o', markersize=7, label='Cancel Rate (%)')
for i, (idx, row) in enumerate(bs.iterrows()):
    ax2.text(i, row['rate']+1.2, f"{row['rate']}%",
             ha='center', fontsize=9, fontweight='bold', color=C_RED)
ax2.set_ylabel('Cancellation Rate (%)', color=C_RED)
ax2.tick_params(axis='y', labelcolor=C_RED)
ax2.set_ylim(0, bs['rate'].max()*1.3)

lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbl1+lbl2, frameon=False, loc='upper left')
ax1.set_title('Lead Time vs Cancellation Rate')
ax1.set_xlabel('Lead Time (days before arrival)')
plt.tight_layout()
plt.show()

## 9. Business Insights

All values below come directly from the dataset.

| # | Insight | Key Figure | Recommended Action |
|---|---|---|---|
| 1 | High cancellation rate across both hotels | **27.72%** (24,025 bookings) | Tiered cancellation policy; incentivise non-refundable bookings |
| 2 | City Hotel cancels 6.51 pp more than Resort | **30.24%** vs **23.73%** | Stricter overbooking buffer for City Hotel |
| 3 | Longer lead time → higher cancellation | 9.77% (0–7 days) to **41.10%** (365+ days), r = 0.183 | Flag long-lead bookings; offer early-bird non-refundable discounts |
| 4 | Extreme seasonality — 2.4× peak vs trough | Aug: **11,195** vs Jan: **4,642** | Dynamic pricing peaks in Jul–Aug; off-peak promotions Nov–Feb |
| 5 | OTAs control 59.18% of bookings | **51,300** via Online TA | Book-direct campaign with best-rate guarantee |
| 6 | Near-zero repeat guest rate | Only **3.63%** (3,147 guests) | Post-checkout loyalty email with 10% direct discount |
| 7 | Resort guests stay 41% longer | **4.44** vs **3.15** nights | Minimum-stay packages for Resort; express services for City |
| 8 | City Hotel higher ADR but higher risk | €112.18 (₹10,096) vs €100.49 (₹9,044) | Differentiated overbooking: City 8–10%, Resort 5–6% |

## 10. Conclusion

This analysis of **86,678 hotel bookings** (after cleaning 119,390 raw records) across City Hotel and Resort Hotel from 2015–2017 reveals several critical operational and commercial findings:

**Cancellation risk is the most urgent issue.** 27.72% of all bookings do not result in a stay. City Hotel's 30.24% rate is significantly higher than Resort Hotel's 23.73%. Lead time is the strongest available predictor — same-week bookings cancel at only 9.77% while 365+ day bookings cancel at 41.10% (Pearson r = 0.183).

**Demand is highly seasonal.** August peaks at 11,195 bookings — 2.4× January's 4,642. Dynamic pricing and targeted off-peak promotions are essential to improve year-round revenue.

**Channel dependency is a structural risk.** Online Travel Agents account for 59.18% of bookings, carrying 15–20% commission costs. Direct bookings (13.45%) should be grown through loyalty incentives and best-rate guarantees.

**Guest retention is nearly absent.** Only 3.63% of bookings are from repeat guests, indicating significant untapped value in post-stay retention programmes.

**The two hotel types are fundamentally different products.** Resort Hotel guests stay 41% longer (4.44 vs 3.15 nights), have lower cancellation rates, and a stronger weekend profile — indicating leisure travellers. City Hotel guests are shorter-stay, weekday-heavy, and higher-cancellation — indicating business and transit travellers. Strategy, pricing, and policies should be calibrated separately for each.